In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import warnings


In [19]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')
print(f"Train - {train.shape}")
print(f"Test  - {test.shape}")
  


Train - (15000, 14)
Test  - (10000, 13)


In [20]:

def feature_engineering(df):
    df = df.copy()
    

    df['Surname_Length'] = df['Surname'].apply(lambda x: len(str(x)))
    
    
    df['IsBalanceZero'] = (df['Balance'] == 0).astype(int)
    
    
    df['Balance_per_Product'] = df['Balance'] / (df['NumOfProducts'] + 1e-6)
    df['Balance_to_Salary'] = df['Balance'] / (df['EstimatedSalary'] + 1e-6)
    df['Tenure_to_Age'] = df['Tenure'] / (df['Age'] + 1e-6)
    df['Age_per_Product'] = df['Age'] / (df['NumOfProducts'] + 1e-6)
    
    
    df = df.drop(['id', 'CustomerId'], axis=1, errors='ignore')
    
    return df

print("gen features...")
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)


cat_cols = ['Surname', 'Geography', 'Gender']
for col in cat_cols:
    train_fe[col] = train_fe[col].astype('category')
    test_fe[col] = test_fe[col].astype('category')

X = train_fe.drop('Exited', axis=1)
y = train_fe['Exited']
test_X = test_fe

gen features...


In [21]:
n_splits = 15
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cb_preds = np.zeros(len(test_X))
lgb_preds = np.zeros(len(test_X))
xgb_preds = np.zeros(len(test_X))

print("Starting model training...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"--- Fold {fold + 1}/{n_splits} ---")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
  
    cb_model = cb.CatBoostClassifier(
        iterations=1500,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        cat_features=cat_cols,
        verbose=0
    )
    cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    cb_preds += cb_model.predict_proba(test_X)[:, 1] / n_splits
    

    lgb_model = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=7,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_preds += lgb_model.predict_proba(test_X)[:, 1] / n_splits


    X_train_xgb, X_val_xgb, test_X_xgb = X_train.copy(), X_val.copy(), test_X.copy()
    for c in cat_cols:
        X_train_xgb[c] = X_train_xgb[c].cat.codes
        X_val_xgb[c] = X_val_xgb[c].cat.codes
        test_X_xgb[c] = test_X_xgb[c].cat.codes
        
    xgb_model = xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=42,
        early_stopping_rounds=100
    )
    xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_val_xgb, y_val)], verbose=0)
    xgb_preds += xgb_model.predict_proba(test_X_xgb)[:, 1] / n_splits

Starting model training...
--- Fold 1/15 ---


C:\Users\tavus\Desktop\ml 10\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


--- Fold 2/15 ---


C:\Users\tavus\Desktop\ml 10\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


--- Fold 3/15 ---


...

C:\Users\tavus\Desktop\ml 10\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


--- Fold 14/15 ---


C:\Users\tavus\Desktop\ml 10\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


--- Fold 15/15 ---


C:\Users\tavus\Desktop\ml 10\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [24]:
final_preds = (cb_preds * 0.45) + (lgb_preds * 0.30) + (xgb_preds * 0.25)

sample_sub['Exited'] = final_preds
sample_sub.to_csv('submission.csv', index=False)
print("submission.csv created r for submis")

submission.csv created r for submis
